# LeetCode 2037 — Minimum Number of Moves to Seat Everyone bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-moves-to-seat-everyone/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Sorting, Counting Sort

```python
class Solution:
    def minMovesToSeat(self, seats: List[int], students: List[int]) -> int:

```

**Mục tiêu học**

1. Đổi bài toán "di chuyển từng bước ±1" thành bài toán **ghép cặp** (matching) học sinh ↔ ghế với chi phí `|ghế - học sinh|`.
2. Hiểu `abs`, `sorted` / `list.sort`, `zip`, `sum` với generator, `itertools.permutations`.
3. Viết brute force thử mọi cách ghép để thấy bài toán, rồi chứng minh **greedy sau khi sắp xếp** là tối ưu (lập luận "đổi chỗ / exchange argument").
4. Cài đặt thêm cách **counting sort** tận dụng giá trị ≤ 100.
5. Nộp `class Solution`, kiểm thử đối chiếu và làm bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Có `n` ghế và `n` học sinh trên một đường thẳng. `seats[i]` là vị trí ghế thứ `i`, `students[j]` là vị trí học sinh thứ `j`. Mỗi **move** đưa một học sinh dịch sang trái hoặc phải 1 đơn vị. Trả về **tổng số move nhỏ nhất** để mỗi học sinh ngồi một ghế, không hai học sinh chung ghế. (Ban đầu có thể có nhiều ghế/học sinh cùng vị trí.)

| Ví dụ | seats | students | Output |
|---|---|---|---|
| 1 | `[3,1,5]` | `[2,7,4]` | `4` |
| 2 | `[4,1,5,9]` | `[1,3,2,6]` | `7` |
| 3 | `[2,2,6,6]` | `[1,3,2,6]` | `4` |

Constraints: `n == seats.length == students.length`, `1 <= n <= 100`, `1 <= seats[i], students[j] <= 100`.

**Quan sát đầu tiên:** đưa học sinh ở vị trí `p` tới ghế ở vị trí `q` tốn đúng `|p - q|` move (đi thẳng, không vòng). Học sinh không cản nhau (có thể đứng chung vị trí khi đang đi). Vậy bài toán là: **ghép mỗi học sinh với một ghế** sao cho tổng `|p - q|` nhỏ nhất.

In [1]:
# Ví dụ 1 theo cách ghép trong đề: (2->1), (7->5), (4->3)
pairs = [(2, 1), (7, 5), (4, 3)]
for p, q in pairs:
    print(f"học sinh {p} -> ghế {q}: {abs(p - q)} move")
print("Tổng:", sum(abs(p - q) for p, q in pairs))

học sinh 2 -> ghế 1: 1 move
học sinh 7 -> ghế 5: 2 move
học sinh 4 -> ghế 3: 1 move
Tổng: 4


## 2. Nền tảng Python

### 2.1. `abs(x)` — giá trị tuyệt đối

Khoảng cách giữa hai điểm trên trục số là `abs(a - b)`, không phụ thuộc ai đứng bên trái.

In [2]:
print(abs(2 - 7), abs(7 - 2))   # 5 5
print(abs(-3), abs(0))           # 3 0

5 5
3 0


### 2.2. `sorted(lst)` và `lst.sort()`

- `sorted(lst)` trả về **list mới** đã sắp xếp tăng, list gốc giữ nguyên.
- `lst.sort()` sắp xếp **tại chỗ** (in-place), trả về `None`.
- Cả hai tốn `O(n log n)`.

Lỗi kinh điển: `lst = lst.sort()` → `lst` thành `None`.

In [3]:
seats = [3, 1, 5]
print(sorted(seats), seats)     # [1, 3, 5] [3, 1, 5]  -> gốc không đổi
x = seats.sort()
print(x, seats)                 # None [1, 3, 5]       -> gốc bị đổi

[1, 3, 5] [3, 1, 5]
None [1, 3, 5]


### 2.3. `zip(a, b)` — đi song song hai list

`zip([1,3,5], [2,4,7])` cho các cặp `(1,2), (3,4), (5,7)`: phần tử cùng vị trí ghép với nhau. Dừng ở list ngắn hơn.

In [4]:
for q, p in zip([1, 3, 5], [2, 4, 7]):
    print("ghế", q, "<-> học sinh", p)
print(list(zip("abc", [1, 2])))   # dừng ở list ngắn hơn: [('a', 1), ('b', 2)]

ghế 1 <-> học sinh 2
ghế 3 <-> học sinh 4
ghế 5 <-> học sinh 7
[('a', 1), ('b', 2)]


### 2.4. `sum(...)` với generator expression

`sum(abs(q - p) for q, p in zip(A, B))` đọc là "cộng `abs(q-p)` cho mọi cặp". Không cần tạo list trung gian.

In [5]:
A, Bv = [1, 3, 5], [2, 4, 7]
print(sum(abs(q - p) for q, p in zip(A, Bv)))   # 1 + 1 + 2 = 4

4


### 2.5. `itertools.permutations` — mọi hoán vị (cho brute force)

`permutations([a, b, c])` sinh mọi thứ tự sắp xếp: 3! = 6 thứ tự. Với ghép cặp: cố định thứ tự học sinh, thử mọi hoán vị của ghế → mọi cách ghép.

In [6]:
from itertools import permutations
perms = list(permutations([1, 3, 5]))
print(len(perms))
for p in perms:
    print(p)

6
(1, 3, 5)
(1, 5, 3)
(3, 1, 5)
(3, 5, 1)
(5, 1, 3)
(5, 3, 1)


## 3. Cách 1 — Brute force: thử mọi cách ghép

Cố định học sinh theo thứ tự `students`, cho ghế chạy qua mọi hoán vị; với mỗi hoán vị tính tổng chi phí, lấy min.

Số cách ghép là `n!` → chỉ dùng được với `n` rất nhỏ (n = 10 đã là 3.6 triệu). Nhưng nó là **oracle** (đáp án chắc chắn đúng) để kiểm tra cách nhanh.

In [7]:
def min_moves_bruteforce(seats, students):
    best = float("inf")                      # "vô cực": giá trị lớn hơn mọi số
    for perm in permutations(seats):
        cost = sum(abs(q - p) for q, p in zip(perm, students))
        best = min(best, cost)
    return best

print(min_moves_bruteforce([3, 1, 5], [2, 7, 4]))         # 4
print(min_moves_bruteforce([4, 1, 5, 9], [1, 3, 2, 6]))   # 7
print(min_moves_bruteforce([2, 2, 6, 6], [1, 3, 2, 6]))   # 4

4
7
4


Xem toàn bộ 6 cách ghép của ví dụ 1 để thấy cách nào tốt nhất:

In [8]:
students = [2, 7, 4]
for perm in permutations([3, 1, 5]):
    detail = [f"{p}->{q}" for q, p in zip(perm, students)]
    cost = sum(abs(q - p) for q, p in zip(perm, students))
    print(detail, "tổng =", cost)

['2->3', '7->1', '4->5'] tổng = 8
['2->3', '7->5', '4->1'] tổng = 6
['2->1', '7->3', '4->5'] tổng = 6
['2->1', '7->5', '4->3'] tổng = 4
['2->5', '7->3', '4->1'] tổng = 10
['2->5', '7->1', '4->3'] tổng = 10


## 4. Nút thắt và insight

`n!` cách ghép là quá nhiều khi `n = 100`. Nhìn bảng trên, cách tốt nhất (tổng 4) là `2->1, 7->5, 4->3`. Sắp xếp lại theo vị trí:

- học sinh sắp xếp: `2, 4, 7`
- ghế sắp xếp: `1, 3, 5`

→ học sinh **nhỏ thứ k** ngồi ghế **nhỏ thứ k**. Không có "chéo" nhau!

**Insight (greedy):** sắp xếp cả hai mảng rồi ghép theo vị trí `i`. Lý do trực giác: nếu hai học sinh `a < b` ngồi chéo vào hai ghế `x > y` (a→x, b→y), đổi lại (a→y, b→x) **không bao giờ tệ hơn**. Lặp lại việc "gỡ chéo" ta được cách ghép đã sắp xếp.

In [9]:
# Kiểm tra "gỡ chéo không tệ hơn" trên vài bộ số
for a, b_, y, x in [(2, 7, 3, 5), (1, 2, 5, 6), (1, 10, 4, 5), (3, 3, 1, 9)]:
    cheo = abs(a - x) + abs(b_ - y)        # a<=b ngồi ghế lớn x, b ngồi ghế nhỏ y
    thang = abs(a - y) + abs(b_ - x)       # a ngồi ghế nhỏ, b ngồi ghế lớn
    print(f"a={a}, b={b_}, ghế y={y} < x={x}: chéo={cheo}, thẳng={thang}, thẳng<=chéo? {thang <= cheo}")

a=2, b=7, ghế y=3 < x=5: chéo=7, thẳng=3, thẳng<=chéo? True
a=1, b=2, ghế y=5 < x=6: chéo=8, thẳng=8, thẳng<=chéo? True
a=1, b=10, ghế y=4 < x=5: chéo=10, thẳng=8, thẳng<=chéo? True
a=3, b=3, ghế y=1 < x=9: chéo=8, thẳng=8, thẳng<=chéo? True


## 5. Cách 2 (lời giải chính) — Sắp xếp + ghép theo thứ tự

Từng mảnh:

1. `seats_sorted = sorted(seats)`, `students_sorted = sorted(students)`.
2. Ghép `seats_sorted[i]` với `students_sorted[i]` bằng `zip`.
3. Cộng `abs(...)`.

In [10]:
seats, students = [4, 1, 5, 9], [1, 3, 2, 6]
s1, s2 = sorted(seats), sorted(students)
print("ghế đã sắp xếp      :", s1)
print("học sinh đã sắp xếp :", s2)
print("chi phí từng cặp    :", [abs(q - p) for q, p in zip(s1, s2)])

ghế đã sắp xếp      : [1, 4, 5, 9]
học sinh đã sắp xếp : [1, 2, 3, 6]
chi phí từng cặp    : [0, 2, 2, 3]


In [11]:
def min_moves_sort(seats, students):
    return sum(abs(q - p) for q, p in zip(sorted(seats), sorted(students)))

print(min_moves_sort([3, 1, 5], [2, 7, 4]))          # 4
print(min_moves_sort([4, 1, 5, 9], [1, 3, 2, 6]))    # 7
print(min_moves_sort([2, 2, 6, 6], [1, 3, 2, 6]))    # 4

4
7
4


## 6. TRACE từng bước

Ví dụ 3: `seats = [2,2,6,6]`, `students = [1,3,2,6]` → sắp xếp: ghế `[2,2,6,6]`, học sinh `[1,2,3,6]`.

| i | ghế | học sinh | `abs` | tổng tích lũy |
|---|---|---|---|---|
| 0 | 2 | 1 | 1 | 1 |
| 1 | 2 | 2 | 0 | 1 |
| 2 | 6 | 3 | 3 | 4 |
| 3 | 6 | 6 | 0 | 4 |

Kết quả `4`. Chú ý: hai ghế cùng vị trí 2 vẫn là **hai ghế riêng** — sắp xếp giữ cả hai.

In [12]:
def trace_min_moves(seats, students):
    s1, s2 = sorted(seats), sorted(students)
    print("seats   sắp xếp:", s1)
    print("students sắp xếp:", s2)
    total = 0
    for i, (q, p) in enumerate(zip(s1, s2)):
        d = abs(q - p)
        total += d
        print(f"i={i}: ghế {q} <- học sinh {p}, cần {d} move, tổng = {total}")
    return total

trace_min_moves([2, 2, 6, 6], [1, 3, 2, 6])

seats   sắp xếp: [2, 2, 6, 6]
students sắp xếp: [1, 2, 3, 6]
i=0: ghế 2 <- học sinh 1, cần 1 move, tổng = 1
i=1: ghế 2 <- học sinh 2, cần 0 move, tổng = 1
i=2: ghế 6 <- học sinh 3, cần 3 move, tổng = 4
i=3: ghế 6 <- học sinh 6, cần 0 move, tổng = 4


4

In [13]:
trace_min_moves([4, 1, 5, 9], [1, 3, 2, 6])

seats   sắp xếp: [1, 4, 5, 9]
students sắp xếp: [1, 2, 3, 6]
i=0: ghế 1 <- học sinh 1, cần 0 move, tổng = 0
i=1: ghế 4 <- học sinh 2, cần 2 move, tổng = 2
i=2: ghế 5 <- học sinh 3, cần 2 move, tổng = 4
i=3: ghế 9 <- học sinh 6, cần 3 move, tổng = 7


7

## 7. Cách 3 — Counting sort (tận dụng giá trị ≤ 100)

Vì vị trí chỉ từ 1 đến 100, ta không cần `sorted` `O(n log n)`. Dùng **mảng đếm**: `cnt_seat[v]` = số ghế ở vị trí `v`, `cnt_stu[v]` = số học sinh ở `v`. Rồi "trải" lại thành list tăng dần — đó là counting sort, `O(n + V)` với `V = 100`.

In [14]:
def counting_sort(arr, max_val=100):
    cnt = [0] * (max_val + 1)            # cnt[v] = số lần v xuất hiện
    for v in arr:
        cnt[v] += 1
    out = []
    for v in range(max_val + 1):
        out.extend([v] * cnt[v])          # [v]*k tạo list k phần tử v
    return out

print(counting_sort([4, 1, 5, 9, 1]))
print([7] * 3, [7] * 0)

[1, 1, 4, 5, 9]
[7, 7, 7] []


In [15]:
def min_moves_counting(seats, students):
    a = counting_sort(seats)
    b_ = counting_sort(students)
    return sum(abs(q - p) for q, p in zip(a, b_))

print(min_moves_counting([3, 1, 5], [2, 7, 4]))
print(min_moves_counting([4, 1, 5, 9], [1, 3, 2, 6]))
print(min_moves_counting([2, 2, 6, 6], [1, 3, 2, 6]))

4
7
4


## 8. Cách 4 — Hai con trỏ trên mảng đếm (không trải ra list)

Ý tưởng nâng cao: duyệt vị trí từ trái sang phải, giữ `balance` = (số ghế đã gặp) − (số học sinh đã gặp). Mỗi khi bước từ vị trí `v` sang `v+1`, có `|balance|` người (hoặc ghế "đang chờ") phải băng qua đoạn `[v, v+1]` → cộng `|balance|` vào đáp án.

Vì sao? Với cách ghép đã sắp xếp, số cặp có một đầu ≤ v và đầu kia > v đúng bằng `|balance|` tại `v`; mỗi cặp như vậy đóng góp 1 move cho đoạn `[v, v+1]`. Tổng các đoạn = tổng khoảng cách.

In [16]:
def min_moves_balance(seats, students):
    diff = [0] * 102
    for q in seats:
        diff[q] += 1
    for p in students:
        diff[p] -= 1
    total = balance = 0
    for v in range(1, 101):
        balance += diff[v]          # ghế - học sinh tính đến vị trí v
        total += abs(balance)       # số người phải đi qua đoạn [v, v+1]
    return total

print(min_moves_balance([3, 1, 5], [2, 7, 4]))
print(min_moves_balance([4, 1, 5, 9], [1, 3, 2, 6]))
print(min_moves_balance([2, 2, 6, 6], [1, 3, 2, 6]))

4
7
4


In [17]:
def trace_balance(seats, students, lo=1, hi=8):
    diff = [0] * 102
    for q in seats: diff[q] += 1
    for p in students: diff[p] -= 1
    total = balance = 0
    for v in range(lo, hi + 1):
        balance += diff[v]
        total += abs(balance)
        print(f"v={v}: diff={diff[v]:+d}, balance={balance:+d}, cộng {abs(balance)}, total={total}")

trace_balance([3, 1, 5], [2, 7, 4])

v=1: diff=+1, balance=+1, cộng 1, total=1
v=2: diff=-1, balance=+0, cộng 0, total=1
v=3: diff=+1, balance=+1, cộng 1, total=2
v=4: diff=-1, balance=+0, cộng 0, total=2
v=5: diff=+1, balance=+1, cộng 1, total=3
v=6: diff=+0, balance=+1, cộng 1, total=4
v=7: diff=-1, balance=+0, cộng 0, total=4
v=8: diff=+0, balance=+0, cộng 0, total=4


Đọc trace: tại `v=1` có một ghế "dư" (balance +1) → học sinh ở 2 phải đi qua đoạn [1,2] để tới ghế 1. Tại `v=5` và `v=6` balance vẫn +1 → ghế 5 đang chờ học sinh ở 7, người này đi qua đoạn [6,7] và [5,6]. Cộng các đoạn: 1 + 1 + 1 + 1 = 4, khớp đáp án.

## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def minMovesToSeat(self, seats: List[int], students: List[int]) -> int:
        seats.sort()
        students.sort()
        return sum(abs(q - p) for q, p in zip(seats, students))

print(Solution().minMovesToSeat([3, 1, 5], [2, 7, 4]))

4


Lưu ý: bản trên sắp xếp **tại chỗ**, làm thay đổi list đầu vào — LeetCode chấp nhận. Trong code thật, nếu không muốn đổi input hãy dùng `sorted(...)`. Bản counting sort cũng nộp được:

In [19]:
class SolutionCounting:
    def minMovesToSeat(self, seats: List[int], students: List[int]) -> int:
        diff = [0] * 102
        for q in seats:
            diff[q] += 1
        for p in students:
            diff[p] -= 1
        total = balance = 0
        for v in range(1, 101):
            balance += diff[v]
            total += abs(balance)
        return total

print(SolutionCounting().minMovesToSeat([4, 1, 5, 9], [1, 3, 2, 6]))

7


## 10. Vì sao greedy đúng? (exchange argument)

**Bổ đề:** với `a ≤ b` (học sinh) và `y ≤ x` (ghế): `|a − y| + |b − x| ≤ |a − x| + |b − y|`.

Chứng minh ngắn: xét hàm `f(t) = |t|` là hàm lồi. Đặt các điểm trên trục số; ghép "thẳng" (nhỏ–nhỏ, lớn–lớn) thì hai đoạn `[a,y]` và `[b,x]` không bao giờ phủ cùng một đoạn nhỏ nhiều lần hơn ghép "chéo". Kiểm tra mọi thứ tự tương đối của 4 điểm (đã chạy số ở mục 4 và test ngẫu nhiên dưới đây) đều cho bất đẳng thức đúng.

**Từ bổ đề ra kết luận:** lấy một cách ghép tối ưu bất kỳ. Nếu còn cặp "chéo", đổi chéo thành thẳng — tổng không tăng. Mỗi lần đổi giảm số cặp chéo (số nghịch thế), nên sau hữu hạn bước được cách ghép đã sắp xếp với tổng **≤ tối ưu** → nó cũng tối ưu.

**Invariant khi cài đặt:** sau khi xử lý cặp `i`, `total` = tổng chi phí của `i+1` cặp nhỏ nhất đã ghép thẳng.

In [20]:
import random
random.seed(1)
for _ in range(10000):
    a, b_ = sorted(random.randint(1, 20) for _ in range(2))
    y, x = sorted(random.randint(1, 20) for _ in range(2))
    assert abs(a - y) + abs(b_ - x) <= abs(a - x) + abs(b_ - y)
print("Bổ đề gỡ chéo đúng trên 10000 bộ ngẫu nhiên")

Bổ đề gỡ chéo đúng trên 10000 bộ ngẫu nhiên


## 11. Độ phức tạp

Gọi `n` = số ghế = số học sinh, `V = 100` = miền giá trị.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force hoán vị | `O(n! · n)` | `O(n)` | chỉ để kiểm tra, n ≤ 8 |
| 2. Sắp xếp + zip | `O(n log n)` | `O(n)` với `sorted` (Timsort có thể cần bộ nhớ phụ), `O(1)` thêm nếu `.sort()` không tính bộ nhớ sắp xếp | lời giải chuẩn |
| 3. Counting sort | `O(n + V)` | `O(n + V)` | tận dụng giá trị nhỏ |
| 4. Balance trên mảng hiệu | `O(n + V)` | `O(V)` | không cần trải list |

## 12. Bộ test

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS ({actual})")

sol = Solution()
cases = [
    ("ví dụ 1", [3, 1, 5], [2, 7, 4], 4),
    ("ví dụ 2", [4, 1, 5, 9], [1, 3, 2, 6], 7),
    ("ví dụ 3 (ghế trùng vị trí)", [2, 2, 6, 6], [1, 3, 2, 6], 4),
    ("n = 1, cùng chỗ", [5], [5], 0),
    ("n = 1, xa nhất", [1], [100], 99),
    ("đã ngồi đúng sẵn", [1, 2, 3], [3, 1, 2], 0),
    ("mọi học sinh cùng vị trí", [1, 2, 3], [2, 2, 2], 2),
]
for name, s, t, exp in cases:
    check(name, sol.minMovesToSeat(list(s), list(t)), exp)   # list(...) để không đổi dữ liệu test

ví dụ 1                          PASS (4)
ví dụ 2                          PASS (7)
ví dụ 3 (ghế trùng vị trí)       PASS (4)
n = 1, cùng chỗ                  PASS (0)
n = 1, xa nhất                   PASS (99)
đã ngồi đúng sẵn                 PASS (0)
mọi học sinh cùng vị trí         PASS (2)


In [22]:
random.seed(2037)
for _ in range(400):
    n = random.randint(1, 6)
    seats = [random.randint(1, 15) for _ in range(n)]
    students = [random.randint(1, 15) for _ in range(n)]
    exp = min_moves_bruteforce(seats, students)
    got = [min_moves_sort(seats, students), min_moves_counting(seats, students),
           min_moves_balance(seats, students),
           Solution().minMovesToSeat(list(seats), list(students)),
           SolutionCounting().minMovesToSeat(seats, students)]
    assert all(g == exp for g in got), (seats, students, exp, got)
print("400 bộ ngẫu nhiên: mọi cách khớp brute force — PASS")

400 bộ ngẫu nhiên: mọi cách khớp brute force — PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Không sắp xếp, ghép theo chỉ số gốc

```python
return sum(abs(q - p) for q, p in zip(seats, students))
```

Đây chỉ là **một** cách ghép tùy ý, thường không tối ưu.

In [23]:
def wrong_no_sort(seats, students):
    return sum(abs(q - p) for q, p in zip(seats, students))

print("Sai :", wrong_no_sort([3, 1, 5], [2, 7, 4]))     # 1 + 6 + 1 = 8
print("Đúng:", min_moves_sort([3, 1, 5], [2, 7, 4]))    # 4

Sai : 8
Đúng: 4


### Lỗi 2: Chỉ sắp xếp một mảng

Sắp xếp `seats` mà quên `students` → vẫn ghép tùy ý.

### Lỗi 3: `x = x.sort()`

`list.sort()` trả `None`, nên `zip(None, ...)` báo `TypeError`.

### Lỗi 4: Quên `abs`

`sum(q - p ...)` cho kết quả có thể âm hoặc "triệt tiêu" (thực ra bằng `sum(seats) - sum(students)`), không phải số move.

In [24]:
seats, students = [3, 1, 5], [2, 7, 4]
print("Chỉ sort seats :", sum(abs(q - p) for q, p in zip(sorted(seats), students)))
print("Quên abs       :", sum(q - p for q, p in zip(sorted(seats), sorted(students))))
try:
    x = [3, 1, 2]
    x = x.sort()
    print(list(zip(x, [1, 2, 3])))
except TypeError as e:
    print("TypeError:", e)

Chỉ sort seats : 6
Quên abs       : -4
TypeError: 'NoneType' object is not iterable


### Lỗi 5: Tham lam "mỗi học sinh chọn ghế gần nhất còn trống"

Nghe hợp lý nhưng **sai**: học sinh đi trước có thể "giành" ghế khiến người sau phải đi rất xa.

In [25]:
def wrong_nearest_free(seats, students):
    free = list(seats)
    total = 0
    for p in students:                         # theo thứ tự gốc
        best = min(free, key=lambda q: abs(q - p))   # ghế trống gần nhất
        total += abs(best - p)
        free.remove(best)
    return total

seats, students = [2, 4], [3, 1]
print("Gần nhất (sai):", wrong_nearest_free(seats, students))   # 3->2 (1), 1->4 (3) = 4
print("Đúng          :", min_moves_sort(seats, students))       # 1->2, 3->4 = 2

Gần nhất (sai): 4
Đúng          : 2


`min(free, key=...)` chọn phần tử có giá trị `key` nhỏ nhất; `lambda q: abs(q - p)` là hàm ẩn danh tính khoảng cách tới `p`.

## 14. Mẹo nhận dạng pattern

- "Ghép **n với n** trên **trục số**, chi phí = khoảng cách" → **sắp xếp cả hai rồi ghép theo thứ tự**.
- Thấy "minimum total moves/cost" + Easy + sorting tag → thử exchange argument: "nếu chéo thì đổi được".
- Giá trị nhỏ (≤ 10^5) → counting sort thay `sorted`.
- Bài tương tự: 455 Assign Cookies (ghép greedy sau sắp xếp), 881 Boats to Save People, 2410 Maximum Matching of Players With Trainers, 462 Minimum Moves to Equal Array Elements II (dồn về median).

## 15. Biến thể

1. **Nhiều ghế hơn học sinh** (m ghế ≥ n học sinh): không còn đơn giản là zip — cần DP `dp[i][j]` (i học sinh đầu, j ghế đầu đã sắp xếp). Ghép thẳng vẫn đúng **thứ tự**, nhưng phải chọn **tập con** ghế.
2. **Tất cả học sinh dồn về một điểm** (không ghế): điểm tối ưu là **median** (LeetCode 462).
3. **Trả về cách ghép**, không chỉ tổng: giữ chỉ số gốc khi sắp xếp (`sorted(range(n), key=...)`).

In [26]:
def min_moves_more_seats(seats, students):
    # dp[i][j]: chi phí nhỏ nhất xếp i học sinh đầu vào j ghế đầu (đều đã sắp xếp)
    s, t = sorted(seats), sorted(students)
    m, n = len(s), len(t)
    INF = float("inf")
    dp = [[INF] * (m + 1) for _ in range(n + 1)]
    for j in range(m + 1):
        dp[0][j] = 0
    for i in range(1, n + 1):
        for j in range(i, m + 1):
            dp[i][j] = min(dp[i][j - 1],                          # bỏ ghế j
                           dp[i - 1][j - 1] + abs(s[j - 1] - t[i - 1]))  # học sinh i ngồi ghế j
    return dp[n][m]

print(min_moves_more_seats([1, 5, 9, 10], [4, 10]))   # 4->5 (1), 10->10 (0) = 1

1


## 16. Bài tập tự luyện

**Bài 1 — Tính tay:** `seats = [10, 1, 4]`, `students = [3, 8, 2]`. Sắp xếp hai mảng, lập bảng như mục 6 và tính đáp án.

**Bài 2 — Đoán output:** `sorted([3,1,2])`, `[3,1,2].sort()`, `list(zip([1,2,3],[9,8]))`, `[0]*3`.

**Bài 3 — TODO:** viết `exercise_min_moves(seats, students)` bằng sắp xếp + zip.

**Bài 4 — TODO:** viết `matching_pairs(seats, students)` trả về list các cặp `(vị trí học sinh, vị trí ghế)` theo cách ghép tối ưu (sắp xếp theo học sinh).

**Bài 5 — TODO biến thể:** `min_moves_to_one_point(nums)` — mọi học sinh dồn về **cùng một điểm** nguyên; trả tổng move nhỏ nhất.

**Bài 6 — Mini challenge:** viết `exercise_balance(seats, students)` không dùng sắp xếp, chỉ dùng mảng hiệu `diff` và biến `balance` như mục 8.

In [27]:
def exercise_min_moves(seats, students):
    # TODO 1: sắp xếp hai mảng
    # TODO 2: ghép theo chỉ số bằng zip
    # TODO 3: cộng abs
    pass

# assert exercise_min_moves([3,1,5],[2,7,4]) == 4

In [28]:
def matching_pairs(seats, students):
    # TODO: trả list (học_sinh, ghế) sau khi ghép tối ưu
    pass

# assert matching_pairs([3,1,5],[2,7,4]) == [(2, 1), (4, 3), (7, 5)]

In [29]:
def min_moves_to_one_point(nums):
    # TODO: nghĩ xem điểm nào làm tổng |x - c| nhỏ nhất
    pass

# assert min_moves_to_one_point([1, 2, 3]) == 2
# assert min_moves_to_one_point([1, 10, 2, 9]) == 16

In [30]:
def exercise_balance(seats, students):
    # TODO: diff[q] += 1 cho ghế, diff[p] -= 1 cho học sinh, cộng |balance| mỗi vị trí
    pass

# assert exercise_balance([4,1,5,9],[1,3,2,6]) == 7

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | ghép nhỏ với nhỏ | ghế `[1,4,10]`, học sinh `[2,3,8]` | `1 + 1 + 2` |
| 2 | `sort` trả gì? | zip dừng ở list ngắn | `[1,2,3]`, `None`, `[(1,9),(2,8)]`, `[0,0,0]` |
| 3 | xem mục 5 | `zip(sorted(a), sorted(b))` | `sum(abs(q-p) for ...)` |
| 4 | vẫn sắp xếp hai mảng | ghép `zip(sorted(students), sorted(seats))` | `list(zip(...))` |
| 5 | điểm ở "giữa" | sắp xếp, lấy phần tử giữa (median) | `sum(abs(x - med) for x in nums)` |
| 6 | mục 8 | `diff = [0]*102` | `balance += diff[v]; total += abs(balance)` |

## 18. Đáp án tham khảo

**Bài 1:** ghế `[1,4,10]`, học sinh `[2,3,8]` → `|1-2| + |4-3| + |10-8| = 1 + 1 + 2 = 4`.

**Bài 2:** `[1, 2, 3]` · `None` · `[(1, 9), (2, 8)]` · `[0, 0, 0]`.

**Bài 5 — vì sao median:** nếu điểm `c` có nhiều người bên trái hơn bên phải, dịch `c` sang trái 1 đơn vị làm tổng giảm. Cân bằng khi số người hai bên bằng nhau → median.

In [31]:
print(min_moves_sort([10, 1, 4], [3, 8, 2]))
print(sorted([3, 1, 2]), [3, 1, 2].sort(), list(zip([1, 2, 3], [9, 8])), [0] * 3)

4
[1, 2, 3] None [(1, 9), (2, 8)] [0, 0, 0]


In [32]:
def exercise_min_moves_answer(seats, students):
    return sum(abs(q - p) for q, p in zip(sorted(seats), sorted(students)))

def matching_pairs_answer(seats, students):
    return list(zip(sorted(students), sorted(seats)))

def min_moves_to_one_point_answer(nums):
    s = sorted(nums)
    med = s[len(s) // 2]
    return sum(abs(x - med) for x in s)

def exercise_balance_answer(seats, students):
    diff = [0] * 102
    for q in seats:
        diff[q] += 1
    for p in students:
        diff[p] -= 1
    total = balance = 0
    for v in range(1, 101):
        balance += diff[v]
        total += abs(balance)
    return total

assert exercise_min_moves_answer([3, 1, 5], [2, 7, 4]) == 4
assert matching_pairs_answer([3, 1, 5], [2, 7, 4]) == [(2, 1), (4, 3), (7, 5)]
assert min_moves_to_one_point_answer([1, 2, 3]) == 2
assert min_moves_to_one_point_answer([1, 10, 2, 9]) == 16
assert exercise_balance_answer([4, 1, 5, 9], [1, 3, 2, 6]) == 7
random.seed(5)
for _ in range(300):
    nums = [random.randint(1, 30) for _ in range(random.randint(1, 7))]
    brute = min(sum(abs(x - c) for x in nums) for c in range(1, 31))
    assert min_moves_to_one_point_answer(nums) == brute
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


## 19. Tổng kết — template + checklist

**Template ghép cặp trên trục số:**

```python
a.sort(); b.sort()
return sum(abs(x - y) for x, y in zip(a, b))
```

Nhớ lý do: **ghép chéo luôn gỡ được mà không tệ hơn** → ghép theo thứ tự đã sắp xếp là tối ưu.

**Checklist submit**

- [ ] Sắp xếp **cả hai** mảng?
- [ ] Dùng `abs` cho từng cặp?
- [ ] Không viết `x = x.sort()`?
- [ ] Tên method `minMovesToSeat`, trả về `int`.
- [ ] Đã thử ví dụ có ghế trùng vị trí (ví dụ 3) và n = 1.